# Loading Data

In [2]:
import pandas as pd
import ast
import numpy as np

In [3]:
mbm_bowl = pd.read_csv(r'D:\all_league\all_league_bowling_card.csv')
playing_xi = pd.read_csv(r'D:\all_league\all_league_playing_xi.csv')
player_info = pd.read_csv(r'D:\all_league\people3.csv')

In [4]:
len(mbm_bowl.match_id.unique())

7323

# Cleaning Data

In [5]:
# Standardize bowler_id in mbm_bowl
mbm_bowl['bowler_id'] = mbm_bowl['bowler_id'].astype(str).str.replace(r'\.0$', '', regex=True)


In [6]:
# Convert key_cricinfo to string
player_info['key_cricinfo'] = player_info['key_cricinfo'].astype(str)

# Standardize bowler_id in mbm_bowl
mbm_bowl['bowler_id'] = mbm_bowl['bowler_id'].apply(lambda x: str(int(float(x))) if pd.notnull(x) else x)

# Function to parse playing_xi columns
def parse_playing_xi(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

# Apply the conversion on both team1 and team2 playing XI columns
playing_xi['team1_playing_xi'] = playing_xi['team1_playing_xi'].apply(parse_playing_xi)
playing_xi['team2_playing_xi'] = playing_xi['team2_playing_xi'].apply(parse_playing_xi)

# Ensure all player IDs in playing_xi are strings
playing_xi['team1_playing_xi'] = playing_xi['team1_playing_xi'].apply(lambda x: [str(pid) for pid in x])
playing_xi['team2_playing_xi'] = playing_xi['team2_playing_xi'].apply(lambda x: [str(pid) for pid in x])

# Define a helper function that returns the longest non-null string among the names
def get_longest_name(row):
    names = [row['name'], row['unique_name']]
    names = [name for name in names if pd.notnull(name)]
    return max(names, key=len) if names else ''

# Create the player mapping dictionary
player_map = dict(zip(player_info['key_cricinfo'], player_info.apply(get_longest_name, axis=1)))


# handling dnb values

In [7]:
# Standardize player_map keys for consistent format
standardized_player_map = {}
for key, value in player_map.items():
    clean_key = str(key).replace('.0', '')
    standardized_player_map[clean_key] = value

# Create a function to standardize player IDs consistently
def standardize_id(player_id):
    if pd.isna(player_id):
        return ""
    return str(player_id).replace('.0', '')

dnb_bowl_rows = []

# Group by match_id and innings
for (match_id, inns), group in mbm_bowl.groupby(['match_id', 'inns']):
    # Extract match details
    date = group['date'].iloc[0]
    bat_team = group['bat_team'].iloc[0]
    bowl_team = group['bowl_team'].iloc[0]
    venue = group['venue'].iloc[0]
    league = group['league'].iloc[0]

    # Get set of bowler IDs who actually bowled (standardized format)
    bowled_ids = {standardize_id(bid) for bid in group['bowler_id'].unique()}

    # Lookup the playing XI row for the match_id
    xi_row = playing_xi[playing_xi['match_id'] == match_id]
    if xi_row.empty:
        print(f"Warning: Match ID {match_id} not found in playing XI data.")
        continue

    row = xi_row.iloc[0]

    # Get the playing XI for the bowling team
    if row['team1_name'] == bowl_team:
        full_xi = row['team1_playing_xi']
    elif row['team2_name'] == bowl_team:
        full_xi = row['team2_playing_xi']
    else:
        print(f"Warning: Bowling team {bowl_team} not found in playing XI teams for match {match_id}.")
        continue

    # Find players who didn't bowl (in playing XI but not in bowled_ids)
    dnb_bowlers = [pid for pid in full_xi if standardize_id(pid) not in bowled_ids]

    # Add a new row for each bowler who did not bowl
    for pid in dnb_bowlers:
        # Get the bowler's name from the player map
        bowler_name = standardized_player_map.get(standardize_id(pid), "")

        new_row = {
            'date': date,
            'match_id': match_id,
            'bowl_team': bowl_team,
            'bat_team': bat_team,
            'bowler_id': pid,
            'bowler': bowler_name,
            'inns': inns,
            'overs': 'DNB',
            'runs': 0,
            'wickets': 0,
            'maidens': 0,
            'eco': 0,
            'balls': 0,
            'lbw_or_bowled': 0,
            'dots': 0,
            'venue': venue,
            'league': league  # Include league in the new row
        }
        dnb_bowl_rows.append(new_row)

In [8]:
# Create DataFrame for DNB bowlers
dnb_df = pd.DataFrame(dnb_bowl_rows)

In [10]:
# Combine the original and DNB DataFrames
combined_df = pd.concat([mbm_bowl, dnb_df], ignore_index=True)

In [11]:
# Also update this section to use the standardized player map
combined_df['bowler'] = combined_df.apply(
    lambda row: row['bowler'] if pd.notnull(row['bowler']) and row['bowler'] != ""
    else standardized_player_map.get(str(row['bowler_id']).replace('.0', ''), ""),
    axis=1
)
# Sort and save the combined DataFrame
combined_df['is_DNB_Bowl'] = combined_df['overs'].apply(lambda x: 1 if x == "DNB" else 0)
combined_df = combined_df.sort_values(by=['match_id', 'inns', 'is_DNB_Bowl']).reset_index(drop=True)
combined_df.drop(columns=['is_DNB_Bowl'], inplace=True)

In [12]:
combined_df


,match_id,bowler,bowler_id,bowl_team,bat_team,inns,overs,balls,wickets,runs,lbw_or_bowled,dots,eco,maidens,venue,date,league
0,211028,B Lee,6278,Australia,England,1,3.0,21,0,31.0,0,6,10.333333,0,The Rose Bowl,2005-06-13,T20IS
1,211028,GD McGrath,6565,Australia,England,1,4.0,25,3,31.0,0,10,7.750000,0,The Rose Bowl,2005-06-13,T20IS
2,211028,MS Kasprowicz,6126,Australia,England,1,3.0,18,1,28.0,0,2,9.333333,0,The Rose Bowl,2005-06-13,T20IS
3,211028,JN Gillespie,5392,Australia,England,1,4.0,25,1,49.0,1,5,12.250000,0,The Rose Bowl,2005-06-13,T20IS
4,211028,MJ Clarke,4578,Australia,England,1,3.0,18,1,25.0,0,4,8.333333,0,The Rose Bowl,2005-06-13,T20IS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161036,1476798,NR Kirton,820685,Canada,Namibia,2,1.0,6,1,12.0,0,1,12.000000,0,Namibia Cricket Ground,2025-03-23,T20IS
161037,1476798,DS Bajwa,1229251,Canada,Namibia,2,DNB,0,0,0.0,0,0,0.000000,0,Namibia Cricket Ground,2025-03-23,T20IS
161038,1476798,K Tathgur,1206188,Canada,Namibia,2,DNB,0,0,0.0,0,0,0.000000,0,Namibia Cricket Ground,2025-03-23,T20IS
161039,1476798,NS Dhaliwal,817069,Canada,Namibia,2,DNB,0,0,0.0,0,0,0.000000,0,Namibia Cricket Ground,2025-03-23,T20IS


In [175]:
combined_df.to_csv(r'D:\ED\FIFS_IPLungit\data\mbm\final_t20i_mbm_bowl.csv', index=False)